# 1. Imports

```python id="j2ibx0"
import os 
import streamlit as st
```

* `os` → access environment variables
* `streamlit` → frontend web UI

---

```python id="0rv9qj"
from dotenv import load_dotenv
```

Loads `.env` variables like:

* `GROQ_API_KEY`
* `HF_TOKEN`

without hardcoding them.

---

# 2. Text Splitting

```python id="5r48bz"
from langchain.text_splitter import RecursiveCharacterTextSplitter
```

LLMs cannot process huge PDFs directly.

So we split documents into chunks:

Example:

```text id="o3m5cl"
100-page PDF
↓
chunk 1
chunk 2
chunk 3
...
```

Why?

Because embeddings + retrieval work chunk-by-chunk.

---

# 3. Retrieval Chain

```python id="2d6jlwm"
from langchain.chains import (
    create_retrieval_chain,
    create_history_aware_retriever
)
```

These are the CORE of your RAG system.

---

## `create_retrieval_chain`

This creates:

```text id="2f17vl"
User Question
     ↓
Retriever fetches relevant chunks
     ↓
LLM gets context + question
     ↓
Final Answer
```

This is standard RAG.

---

## `create_history_aware_retriever`

This is advanced conversational RAG.

Without this:

```text id="tjlwmm"
User: Who is Elon Musk?
Bot: CEO of Tesla.

User: What company does he run?
```

Retriever sees:

```text id="6u7bl0"
"What company does he run?"
```

Problem:

* “he” is ambiguous
* retriever gets confused

So LangChain first rewrites it into:

```text id="cwyz2q"
"What company does Elon Musk run?"
```

THEN retrieval happens.

This is why it's called:

* history-aware retriever

Very important concept.

---

# 4. Document QA Chain

```python id="7mt5x6"
from langchain.chains.combine_documents import create_stuff_documents_chain
```

After retrieval:

```text id="x4yyye"
Top relevant chunks retrieved
```

Now:

* how do we combine them into prompt context?

This chain:

* stuffs retrieved chunks into prompt

Example:

```text id="2lqx2f"
Context:
Chunk 1
Chunk 2
Chunk 3

Question:
What is RAG?
```

Then sends to LLM.

---

# 5. Vector Database

```python id="ngjlwm"
from langchain_chroma import Chroma
```

This stores embeddings.

Flow:

```text id="kj9qwg"
PDF Text
   ↓
Embeddings
   ↓
Stored in ChromaDB
```

Later:

```text id="hch25n"
Question embedding
   ↓
Similarity search
   ↓
Most relevant chunks returned
```

This is semantic search.

---

# 6. Chat Message History

```python id="pnl4t2"
from langchain_community.chat_message_histories import ChatMessageHistory
```

Stores conversation memory.

Example:

```text id="mjlwmv"
User: What is AI?
Bot: ...

User: Explain it simply.
```

Without history:

* second question makes no sense

With history:

* context persists

---

# 7. Prompt Templates

```python id="dzjlwm"
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)
```

---

## `ChatPromptTemplate`

Dynamic prompts.

Example:

```python id="xtcxha"
("human", "{input}")
```

`{input}` gets replaced dynamically.

---

## `MessagesPlaceholder`

Injects previous chat history.

Without this:

* no memory inside prompt

---

# 8. RunnableWithMessageHistory

```python id="jlwmc0"
from langchain_core.runnables.history import RunnableWithMessageHistory
```

This is what makes your chatbot STATEFUL.

Without this:

* every request becomes isolated

With this:

```text id="w4svry"
Session A → own memory
Session B → separate memory
```

Real production concept.

---

# 9. Embeddings

```python id="jtrmwn"
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
```

Embeddings convert text → vectors.

Example:

```text id="jlwm8p"
"dog" → [0.23, -0.91, ...]
"puppy" → [0.22, -0.88, ...]
```

Semantically similar words become mathematically close.

That’s how retrieval works.

---

# 10. Streamlit UI

```python id="jjlwmn"
st.title()
st.write()
st.text_input()
st.file_uploader()
```

Frontend layer.

---

# 11. PDF Upload Flow

```python id="i5p2r7"
uploaded_file = st.file_uploader(...)
```

User uploads PDF.

Then:

```python id="hjlwm6"
with open(temp_pdf, "wb")
```

temporarily saves it.

---

# 12. PDF Loader

```python id="7pxjlwm"
loader = PyPDFLoader(temp_pdf)
documents = loader.load()
```

Extracts PDF text into LangChain Document objects.

---

# 13. Splitting

```python id="o4y2i7"
text_splitter.split_documents(documents)
```

Large text becomes manageable chunks.

---

# 14. Vector Store Creation

```python id="jlwm2n"
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)
```

Creates:

* embeddings
* vector DB

in one step.

---

# 15. Retriever

```python id="jlwm3k"
retriever = vectorstore.as_retriever()
```

Turns vector DB into searchable retriever.

---

# 16. Contextualization Prompt

```python id="jlwm9z"
contextualize_q_system_prompt
```

This prompt rewrites ambiguous questions.

Very important for conversational RAG.

---

# 17. QA Prompt

```python id="jlwmqa"
system_prompt
```

Controls:

* answer style
* answer length
* hallucination behavior

This is prompt engineering.

---

# 18. Retrieval + QA Pipeline

```python id="s0jlwm"
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
```

Full pipeline:

```text id="jlwm4s"
Question
↓
Rewrite using history
↓
Retrieve chunks
↓
Inject context
↓
LLM answer
```

This is COMPLETE conversational RAG architecture.

---

# 19. Session Memory

```python id="jlwm5q"
st.session_state.store
```

Stores different user conversations.

Example:

```text id="jlwm7f"
session_1 → memory
session_2 → separate memory
```

This simulates multi-user production systems.

---

# 20. RunnableWithMessageHistory

```python id="jlwm1a"
conversational_rag_chain = RunnableWithMessageHistory(...)
```

This binds:

* chain
* session memory
* chat history

into one conversational system.

---

# 21. Invocation

```python id="8qk9u9"
response = conversational_rag_chain.invoke(...)
```

This executes entire pipeline.

---

# Final Architecture

Your app now works like this:

```text id="jlwm6x"
User uploads PDF
        ↓
PDF Loader extracts text
        ↓
Text Splitter creates chunks
        ↓
Embeddings generated
        ↓
Stored in Chroma Vector DB
        ↓
User asks question
        ↓
Question rewritten using history
        ↓
Retriever finds relevant chunks
        ↓
Chunks injected into prompt
        ↓
Groq LLM generates answer
        ↓
Chat history stored
        ↓
Conversation continues contextually
